In [10]:
import pandas as pd

tabular = pd.read_csv("data/train/train_tabular.csv")
notes = pd.read_csv("data/train/train_notes.csv")

merged = pd.merge(
    tabular, notes, 
    how = "inner",
    on = "flight_id"
)

merged["text_data"] = merged["maintenance_note"].map({
    value: failure_rate for value, failure_rate in zip(
        merged["maintenance_note"].unique(),
        merged.groupby("maintenance_note")["failure_within_horizon"].mean()
    )
})
merged.drop(columns=["maintenance_note"], inplace=True)

merged.fillna({"text_data": 0.5}, inplace=True)
merged.to_csv("data/train/train_tabular_notes.csv", index=False)

In [7]:
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression

In [32]:
X = merged.drop(columns=["flight_id", "drone_id", "failure_within_horizon", "failure_mode"])
y = merged["failure_within_horizon"]
g = merged["drone_id"]

In [33]:
categorical_columns = ["model", "motor_type", "firmware_version", "operator_region"]
numerical_columns = X.select_dtypes(include="number").columns

In [34]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_columns),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_columns)
    ]
)
pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000))
    ]
)

In [35]:
models = [
    LogisticRegression(max_iter=1000),
    RandomForestClassifier(n_estimators=100, random_state=42)
]

In [36]:
X

,model,motor_type,firmware_version,battery_capacity_mAh,max_payload_g,propeller_in,manufacture_batch,operator_region,flight_index,payload_g,ambient_temp_C,wind_speed_ms,flight_duration_min,avg_throttle,num_aggressive_maneuvers,cumulative_flight_hours,battery_cycles,text_data
0,C,M3,v3.1,5200,900.0,11.0,3,east,1,69.2,18.75,15.63,14.85,0.341,0,16.44,25,0.968254
1,C,M3,v3.1,5200,900.0,11.0,3,east,2,728.1,29.90,5.97,10.59,0.762,1,16.61,26,0.432432
2,C,M3,v3.1,5200,900.0,11.0,3,east,3,868.3,30.96,12.01,18.62,0.928,2,16.92,27,0.432432
3,C,M3,v3.1,5200,900.0,11.0,3,east,4,840.1,25.32,10.94,17.53,0.920,5,17.21,28,0.062444
4,C,M3,v3.1,5200,900.0,11.0,3,east,5,19.3,28.58,7.10,21.09,0.434,1,17.57,29,0.336683
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15867,B,M2,v4.0,8000,500.0,13.0,1,south,20,50.1,26.04,4.78,8.91,0.550,3,12.05,48,0.312500
15868,B,M2,v4.0,8000,500.0,13.0,1,south,21,161.5,26.64,11.72,16.46,0.533,1,12.33,49,0.062444
15869,B,M2,v4.0,8000,500.0,13.0,1,south,22,392.1,35.24,7.84,14.16,0.729,7,12.56,50,0.336683
15870,B,M2,v4.0,8000,500.0,13.0,1,south,23,334.8,22.88,2.56,16.11,0.711,5,12.83,51,0.312500


In [37]:
for model in models:
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    scores = cross_val_score(
        pipeline, X, y, groups=g, cv = GroupKFold(n_splits=5), scoring="average_precision"
    )

    print(f"Model: {model}, Mean: {scores.mean()}, Std: {scores.std()}")

Model: LogisticRegression(max_iter=1000), Mean: 0.35012580723060915, Std: 0.04020512632314359
Model: RandomForestClassifier(random_state=42), Mean: 0.39733376990963937, Std: 0.0448829378016923
